# M27 lab solution: SQL or vectors

**Brief.** Answer the same 20 questions over a product catalogue with CSV RAG (M19) and with the text-to-SQL agent, and record which wins for each question type.

The catalogue is `../data/catalogue.csv`, a fictional Larkspur price list written for this lab; `../data/questions.jsonl` labels each question `lookup`, `filter` or `aggregate` and gives the answer computed from the CSV. A judge prompt, printed below, compares each system's answer with that reference.

The notebook reads only `../data/` and imports only from `src/`. Run it from this folder; the lab README lists the keys it needs.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, str(Path("../../../../src").resolve()))

from helpers import get_embeddings, get_experientiallabs_llm
from lab_kit import is_hit, load_handbook, load_jsonl

DATA = Path("../data")
llm = get_experientiallabs_llm()
print("LLM:", llm.model_name)

In [ ]:
catalogue = pd.read_csv(DATA / "catalogue.csv")
questions = load_jsonl(DATA / "questions.jsonl")
print(catalogue.shape, "catalogue;", len(questions), "questions")
print(pd.Series([q["type"] for q in questions]).value_counts().to_string())

## 1. CSV RAG, as in M19

One document per row, the row written out as `column: value` lines, embedded into an in-memory vector store. The answer step sees the top 5 rows.

In [ ]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore

def row_text(row):
    return "\n".join(f"{col}: {val}" for col, val in row.items() if pd.notna(val) and val != "")

row_docs = [Document(page_content=row_text(r), metadata={"sku": r["sku"]}) for _, r in catalogue.iterrows()]
row_store = InMemoryVectorStore.from_documents(row_docs, get_embeddings())
row_retriever = row_store.as_retriever(search_kwargs={"k": 5})

CSV_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Answer the question about the Larkspur product catalogue using only these catalogue rows. "
               "Prices are monthly, in USD. If the rows are not enough to answer, say so."),
    ("human", "Rows:\n{rows}\n\nQuestion: {question}"),
])
csv_chain = CSV_PROMPT | llm | StrOutputParser()


def csv_rag(question):
    rows = row_retriever.invoke(question)
    return csv_chain.invoke({"rows": "\n\n".join(d.page_content for d in rows), "question": question})


print(csv_rag(questions[0]["question"]))

## 2. The text-to-SQL agent, as in `04b_text2sql_react_agent.ipynb`

The CSV is loaded into a SQLite file in a temporary folder and opened read-only, so the agent's SQL cannot change it. The agent is `create_agent` with the `SQLDatabaseToolkit` tools.

In [ ]:
import sqlite3
import tempfile

from langchain.agents import create_agent
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase

db_path = Path(tempfile.mkdtemp(prefix="m27_")) / "catalogue.db"
with sqlite3.connect(db_path) as con:
    catalogue.to_sql("catalogue", con, index=False)
db = SQLDatabase.from_uri(f"sqlite:///file:{db_path}?mode=ro&uri=true")
tools = SQLDatabaseToolkit(db=db, llm=llm).get_tools()

SQL_SYSTEM = ("You answer questions about the Larkspur product catalogue by querying a SQLite database. "
              "List the tables, read the schema, then write one SELECT query; check it with the query checker before running it. "
              "Prices are monthly, in USD. The available_on column holds plan names separated by semicolons. "
              "Never write INSERT, UPDATE, DELETE or DDL statements. Answer in one or two sentences.")
sql_agent = create_agent(llm, tools, system_prompt=SQL_SYSTEM)


def sql_answer(question):
    result = sql_agent.invoke({"messages": [{"role": "user", "content": question}]}, {"recursion_limit": 25})
    return result["messages"][-1].content


print([t.name for t in tools])
print(sql_answer(questions[0]["question"]))

## 3. Both systems on the 20 questions, scored by a judge

In [ ]:
from pydantic import BaseModel, Field


class Grade(BaseModel):
    correct: bool = Field(description="True if the answer states the reference answer's facts, with nothing contradicting them")
    reason: str


JUDGE_SYSTEM = ("You grade answers against a reference answer. An answer is correct when it states every fact in the reference "
                "(each product or region named, each number to the cent) and contradicts none of them. "
                "Extra correct detail is fine; a missing item from a list makes the answer incorrect.")
judge = ChatPromptTemplate.from_messages([
    ("system", JUDGE_SYSTEM),
    ("human", "Question: {question}\nReference answer: {reference}\nAnswer to grade: {answer}"),
]) | llm.with_structured_output(Grade)
print("Judge prompt (system message):\n" + JUDGE_SYSTEM)

rows = []
for q in questions:
    for system, fn in (("CSV RAG", csv_rag), ("text-to-SQL", sql_answer)):
        answer = fn(q["question"])
        grade = judge.invoke({"question": q["question"], "reference": q["answer"], "answer": answer})
        rows.append({"id": q["id"], "type": q["type"], "system": system, "correct": grade.correct,
                     "answer": answer, "reason": grade.reason})
graded = pd.DataFrame(rows)
print(graded.pivot(index=["id", "type"], columns="system", values="correct").to_string())

## 4. Results

In [ ]:
by_type = graded.pivot_table(index="type", columns="system", values="correct", aggfunc="sum").astype(int)
by_type["questions"] = graded.groupby("type").id.nunique()
by_type["winner"] = by_type.apply(lambda r: "tie" if r["CSV RAG"] == r["text-to-SQL"]
                                  else ("CSV RAG" if r["CSV RAG"] > r["text-to-SQL"] else "text-to-SQL"), axis=1)
by_type.loc["all"] = [graded[graded.system == "CSV RAG"].correct.sum(), graded[graded.system == "text-to-SQL"].correct.sum(),
                      graded.id.nunique(), ""]
print(by_type.to_string())
print("\nWrong answers:")
for _, r in graded[~graded.correct].iterrows():
    print(f"  {r.id} [{r.system}] {r.answer[:160]!r}  ({r.reason})")

## What this shows

- The table answers the brief: correct answers per question type for each system, and which system won each type.
- Top-k retrieval sees 5 of the 25 rows, so a question whose answer depends on rows it did not retrieve (a full list, a count, an average) is out of its reach however good the answer prompt is. SQL computes over every row.
- Read the wrong answers before drawing a rule. A text-to-SQL miss is often a schema detail, such as the semicolon list in `available_on`, that a sentence in the system prompt or a view can fix; a CSV RAG miss on an aggregate cannot be fixed that way.